# 020번 주제별 텍스트 일상 대화 탐색 
# (60대 이상 데이터X -> 사용X)

**목적**: voice-chat 프롬프트용 어르신 SNS 실제 대화체 어휘·말투 패턴 참조

**Drive 경로**: `내드라이브/Dadam_dataSet/020.주제별 텍스트 일상 .../data/Validation/라벨링데이터/`

**데이터 구조**:
- zip 5개 (플랫폼별): KAKAO / FACEBOOK / INSTAGRAM / BAND / NATEON
- 각 zip 내 JSON 200개
- 파일명 패턴: `BAND_11_05.json`, `NATEON_11_02.json`

**활용처**:
- TASK-02: `voice-chat` 어르신 말투·어휘 참조
- 어르신이 SNS에서 실제로 쓰는 표현 패턴 → 시스템 프롬프트 톤 설정 근거

## 처리 흐름
```
Cell 1 → Drive 마운트
Cell 2 → 경로 설정 및 zip 목록 확인
Cell 3 → 샘플 JSON 구조 파악
Cell 4 → 전체 로드 + 어르신 발화 필터링
Cell 5 → 발화 패턴 분석 (어휘·말투·이모티콘)
Cell 6 → voice-chat 참조용 표현 패턴 저장
Cell 7 → 로컬 다운로드
```

## Cell 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2. 경로 설정 및 zip 파일 목록 확인

In [ ]:
import os
import zipfile

MYDRIVE = '/content/drive/MyDrive'
# Validation/라벨링데이터 사용 (5개 zip, 총 ~25MB)
LABEL_DIR = os.path.join(
    MYDRIVE,
    'Dadam_dataSet',
    '020.주제별 텍스트 일상 대화 데이터',
    'data', 'Validation', '라벨링데이터'
)

print('경로 존재:', os.path.exists(LABEL_DIR))

all_zips = sorted([f for f in os.listdir(LABEL_DIR) if f.endswith('.zip')])
print(f'\nzip 파일 총 {len(all_zips)}개')
for z in all_zips:
    size_mb = os.path.getsize(os.path.join(LABEL_DIR, z)) / (1024 * 1024)
    print(f'  {z}  ({size_mb:.1f} MB)')


## Cell 3. 샘플 JSON 구조 파악

zip 1개(BAND)를 열어 JSON 내부 구조 확인.

예상 필드:
- 발화자 정보 (나이·성별)
- 대화 텍스트
- 플랫폼·주제 메타데이터

In [ ]:
import json

# BAND zip으로 JSON 구조 파악
band_zip = 'VL_04. BAND.zip'
sample_zip_path = os.path.join(LABEL_DIR, band_zip)

with zipfile.ZipFile(sample_zip_path, 'r') as z:
    json_files = [f for f in z.namelist() if f.endswith('.json')]
    with z.open(json_files[0]) as jf:
        raw = json.load(jf)

info = raw['info'][0]
annotations = info['annotations']

print(f'subject: {annotations["subject"]}')
print(f'medianame: {info["medianame"]}')
print(f'lines 수: {len(annotations["lines"])}개')
print()

# lines 전체 구조 확인
print('=== lines 상세 (전체) ===')
for i, line in enumerate(annotations['lines']):
    print(f'[{i}] 키: {list(line.keys())}')
    for k, v in line.items():
        print(f'     {k}: {v}')
    if i >= 2:
        print(f'  ...({len(annotations["lines"]) - 3}개 더)')
        break


## Cell 4. 전체 로드 + 어르신 발화 필터링

5개 플랫폼 zip을 모두 로드하여 발화 데이터 수집.

※ Cell 3 구조 확인 후 `extract_row` 함수의 키 이름을 실제 JSON에 맞게 수정.

In [ ]:
import pandas as pd

SENIOR_AGES = {'50대', '60대', '70대', '80대', '90대'}

def extract_lines(raw, zip_fname, jf_name):
    """JSON 1개에서 발화 라인 전체 추출"""
    results = []
    info = raw.get('info', [{}])[0]
    annotations = info.get('annotations', {})
    subject = annotations.get('subject', '')
    medianame = info.get('medianame', '')
    lines = annotations.get('lines', [])

    for line in lines:
        speaker = line.get('speaker', {})
        text = line.get('norm_text', line.get('text', '')).strip()
        if ' : ' in text:
            text = text.split(' : ', 1)[1].strip()
        if not text or len(text) < 5:
            continue
        results.append({
            'zip_file': zip_fname,
            'json_file': jf_name,
            'platform': medianame,
            'subject': subject,
            'utterance': text,
            'length': len(text),
            'age': speaker.get('age', ''),
            'sex': speaker.get('sex', ''),
            'speech_act': line.get('speechAct', ''),
        })
    return results


# Validation 5개 zip 전체 로드 — 주제 목록 파악용
SAMPLES_PER_ZIP = 50
rows = []

for zip_fname in all_zips:
    zip_path = os.path.join(LABEL_DIR, zip_fname)
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            json_files = [f for f in z.namelist() if f.endswith('.json')]
            for jf_name in json_files[:SAMPLES_PER_ZIP]:
                with z.open(jf_name) as jf:
                    raw = json.load(jf)
                rows.extend(extract_lines(raw, zip_fname, jf_name))
    except Exception as e:
        print(f'오류: {zip_fname} — {e}')

df_full = pd.DataFrame(rows)

# 전체 주제 목록 (중복 제거)
all_subjects = sorted(df_full['subject'].unique())
print(f'전체 주제 수: {len(all_subjects)}개\n')
print('=== 주제 목록 전체 ===')
for s in all_subjects:
    count = (df_full['subject'] == s).sum()
    print(f'  {s} ({count}개)')


## Cell 5. extract_row 수정 후 전체 로드

Cell 3·4 출력을 보고 `extract_row_020` 함수를 실제 JSON 구조에 맞게 수정한 뒤 실행.

In [ ]:
import re
from collections import Counter

# 어르신 발화 패턴 분석
# 1. 종결어미 빈도
ENDING_PATTERNS = [
    (r'요\s*$',       '~요'),
    (r'ㅎ{2,}',       'ㅎㅎ(웃음)'),
    (r'ㅋ{2,}',       'ㅋㅋ(웃음)'),
    (r'~+',           '~(물결)'),
    (r'\.{2,}',       '...(말줄임)'),
    (r'죠\s*$|지요\s*$', '~죠/지요'),
    (r'네요\s*$|네\s*$', '~네/네요'),
    (r'다\s*$',       '~다(평서)'),
    (r'\^\^|:D|:)',   '^^ 이모티콘'),
    (r'!!+',          '!!(강조)'),
]

print('=== 종결어미·표현 패턴 (어르신 발화 기준) ===')
total = len(df_senior)
pattern_counts = {}
for pattern, label in ENDING_PATTERNS:
    count = df_senior['utterance'].str.contains(pattern, regex=True, na=False).sum()
    pct = count / total * 100 if total > 0 else 0
    pattern_counts[label] = {'count': int(count), 'pct': round(pct, 1)}
    print(f'  {label:18s}: {count:4d}개 ({pct:.1f}%)')

# 2. 주제별 어르신 발화 분포
print('\n=== 주제별 어르신 발화 수 (상위 15개) ===')
print(df_senior['subject'].value_counts().head(15))

# 3. 대표 발화 30개 선별 (15자 이상, ㅎㅎ/~요/ㅋ 포함 우선)
ORAL_FLAG = r'ㅎ{2,}|ㅋ{2,}|~+|요\s*$|죠\s*$'
df_senior['is_oral'] = df_senior['utterance'].str.contains(ORAL_FLAG, regex=True, na=False)

df_repr = df_senior[
    (df_senior['length'] >= 15) &
    (df_senior['length'] <= 120)
].sort_values(['is_oral', 'length'], ascending=[False, True]).drop_duplicates('utterance').head(30)

print(f'\n=== 대표 어르신 발화 {len(df_repr)}개 ===')
for i, (_, r) in enumerate(df_repr.iterrows(), 1):
    flag = '🎯' if r['is_oral'] else '  '
    print(f'{flag} {i:2d}. [{r["age"]} {r["sex"]}] [{r["subject"]}] {r["utterance"]}')


## Cell 6. 어르신 발화 패턴 분석

50세 이상 발화만 추려 voice-chat 참조용 어휘·말투 패턴 분석:
1. 자주 쓰는 종결어미
2. 이모티콘·특수문자 사용 패턴
3. 줄임말·신조어 vs 정중체 비율
4. 대표 발화 20개 선별

In [ ]:
import json

# voice-chat 참조용 패턴 결과 저장
repr_utterances = [
    {
        'platform': r['platform'],
        'subject': r['subject'],
        'utterance': r['utterance'],
        'age': r['age'],
        'sex': r['sex'],
        'is_oral': bool(r['is_oral']),
        'length': int(r['length']),
    }
    for _, r in df_repr.iterrows()
]

output = {
    'summary': {
        '총발화': len(df_full),
        '어르신발화': len(df_senior),
        '플랫폼': df_full['platform'].unique().tolist(),
        '어르신비율': round(len(df_senior) / len(df_full) * 100, 1),
    },
    'ending_patterns': pattern_counts,
    'representative_utterances': repr_utterances,
    'note': '020번 SNS 일상 대화 — TASK-02 voice-chat 어르신 말투 참조용',
}

# Drive 저장
OUTPUT_DRIVE = '/content/drive/MyDrive/Dadam_dataSet/020_voicechat_patterns.json'
with open(OUTPUT_DRIVE, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f'Drive 저장 완료: {OUTPUT_DRIVE}')
print(f'대표 발화 {len(repr_utterances)}개 / 어르신 발화 {len(df_senior)}개')


## Cell 7. voice-chat 참조용 표현 패턴 저장

산출물 형식:
```json
{
  "summary": { "총발화": N, "어르신발화": N, "플랫폼": [...] },
  "ending_patterns": { "~요": N%, "ㅎㅎ": N%, ... },
  "representative_utterances": [ { "platform": "KAKAO", "utterance": "..." }, ... ]
}
```

In [ ]:
from google.colab import files

LOCAL_PATH = '/content/020_voicechat_patterns.json'
with open(LOCAL_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

files.download(LOCAL_PATH)
print('다운로드 완료 → prompt/fewshot/020_voicechat_patterns.json 에 저장')


## Cell 8. 로컬 다운로드

→ `prompt/fewshot/020_voicechat_patterns.json` 으로 저장